In [0]:
%sql
CREATE TABLE IF NOT EXISTS spotify_etl.raw.spotify_tokens (
  token_name STRING,
  access_token STRING,
  refresh_token STRING,
  expires_at BIGINT,
  updated_at TIMESTAMP
)
USING delta;

In [0]:
import base64
import time
import requests
from pyspark.sql import functions as F

TOKEN_TABLE = "spotify_etl.raw.spotify_tokens"
TOKEN_NAME = "default"

def load_token_state():
    # Încarcă ultimul token salvat din tabelă
    df = spark.table(TOKEN_TABLE).where(F.col("token_name") == TOKEN_NAME)
    row = df.orderBy(F.col("updated_at").desc()).limit(1).collect()
    if not row:
        # Dacă nu există token, returnează valori implicite
        return {"access_token": None, "refresh_token": None, "expires_at": 0}
    r = row[0].asDict()
    return {
        "access_token": r.get("access_token"),
        "refresh_token": r.get("refresh_token"),
        "expires_at": int(r.get("expires_at") or 0)
    }

def save_token_state(state: dict):
    # Șterge token-ul vechi și salvează noul token în tabelă
    spark.sql(f"DELETE FROM {TOKEN_TABLE} WHERE token_name = '{TOKEN_NAME}'")
    spark.createDataFrame([{
        "token_name": TOKEN_NAME,
        "access_token": state.get("access_token"),
        "refresh_token": state.get("refresh_token"),
        "expires_at": int(state.get("expires_at") or 0)
    }]).withColumn("updated_at", F.current_timestamp()).write.mode("append").saveAsTable(TOKEN_TABLE)

class SpotifyAuthError(RuntimeError):
    # Eroare custom pentru autentificare Spotify
    pass

def refresh_spotify_access_token(refresh_token: str, client_id: str, client_secret: str, timeout_s: int = 10) -> dict:
    # Reîmprospătează access_token-ul folosind refresh_token
    token_url = "https://accounts.spotify.com/api/token"
    basic = base64.b64encode(f"{client_id}:{client_secret}".encode("utf-8")).decode("utf-8")

    headers = {
        "Authorization": f"Basic {basic}",
        "Content-Type": "application/x-www-form-urlencoded",
    }
    data = {
        "grant_type": "refresh_token",
        "refresh_token": refresh_token,
    }

    resp = requests.post(token_url, headers=headers, data=data, timeout=timeout_s)
    try:
        payload = resp.json()
    except Exception:
        raise SpotifyAuthError(f"Refresh failed: {resp.text}")

    if resp.status_code != 200:
        raise SpotifyAuthError(f"Refresh failed ({resp.status_code}): {payload}")

    access_token = payload["access_token"]
    expires_in = int(payload.get("expires_in", 3600))
    now = int(time.time())
    expires_at = now + expires_in - 30  # buffer de 30 secunde

    # Spotify adesea NU retrimite refresh_token la refresh
    new_refresh_token = payload.get("refresh_token") or refresh_token

    return {"access_token": access_token, "refresh_token": new_refresh_token, "expires_at": expires_at}

def get_valid_access_token(client_id: str, client_secret: str, refresh_token_fallback: str) -> dict:
    # Returnează un access_token valid, reîmprospătează dacă e expirat
    state = load_token_state()

    # dacă nu avem încă refresh_token în tabelă, îl setăm din secrets
    if not state.get("refresh_token"):
        state["refresh_token"] = refresh_token_fallback

    now = int(time.time())
    if state.get("access_token") and now < int(state.get("expires_at", 0)):
        # Token-ul este încă valid
        return state

    # Token-ul a expirat, îl reîmprospătăm
    refreshed = refresh_spotify_access_token(
        refresh_token=state["refresh_token"],
        client_id=client_id,
        client_secret=client_secret,
    )
    state.update(refreshed)
    save_token_state(state)
    return state

def spotify_api_call(method: str, url: str, client_id: str, client_secret: str, refresh_token_fallback: str, params=None, json=None, timeout_s: int = 10):
    # Efectuează un apel către API-ul Spotify cu token valid
    state = get_valid_access_token(client_id, client_secret, refresh_token_fallback)

    def _req(access_token: str):
        headers = {"Authorization": f"Bearer {access_token}"}
        return requests.request(method, url, headers=headers, params=params, json=json, timeout=timeout_s)

    resp = _req(state["access_token"])

    # dacă token-ul a fost invalidat, retry o singură dată cu refresh
    if resp.status_code == 401:
        refreshed = refresh_spotify_access_token(state["refresh_token"], client_id, client_secret)
        state.update(refreshed)
        save_token_state(state)
        resp = _req(state["access_token"])

    if not resp.ok:
        raise RuntimeError(f"Spotify API error {resp.status_code}: {resp.text}")

    return resp.json()